In [1]:
import os 
import json
import pandas as pd 
import traceback

In [4]:
from langchain_openai import ChatOpenAI

In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
KEY = os.getenv("OPENAI_API_KEY")

In [15]:
llm=ChatOpenAI(api_key=KEY, model="gpt-3.5-turbo", temperature=0.5)

In [17]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x130c25d00>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x130d01490>, root_client=<openai.OpenAI object at 0x130d14f10>, root_async_client=<openai.AsyncOpenAI object at 0x130c25f10>, temperature=0.5, openai_api_key=SecretStr('**********'), openai_proxy='')

In [20]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.callbacks import get_openai_callback

In [21]:
from openai import  OpenAI

In [22]:
import PyPDF2

In [23]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [24]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [25]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [26]:
quiz_chain = LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)

/var/folders/60/7kl34kgx59qcnt93045l71300000gn/T/ipykernel_43227/4109068563.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  quiz_chain = LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)


In [28]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [29]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"], template=TEMPLATE2)

In [30]:
review_chain = LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [31]:
generate_evaulate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True)

In [32]:
file_path = "../data.txt"

In [33]:
file_path

'../data.txt'

In [34]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [36]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied human cognitive systems contri

In [38]:
# serialise python dict into JSON-format string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [41]:
NUMBER=5
SUBJECT="machine learning"
TONE="simple"

In [43]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaulate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )

/var/folders/60/7kl34kgx59qcnt93045l71300000gn/T/ipykernel_43227/4222452570.py:5: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  response=generate_evaulate_chain(
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Prompt after formatting:

Text:The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
Prompt after formatting:

You are an expert english grammarian and writer. Given a Multiple Choice Quiz for machine learning students.You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:

{
    "1": {
        "mcq": "What term was coined in 1959 by Arthur Samuel to describe the field of computer gaming and artificial intelligence?",
        "options": {
            "a": "Artificial Intelligence",
            "b": "Machine Learning",
            "c": "Neural Networks",
            "d": "Deep Learning"
        },
        "correct": "b"
    },
    "2": {
        "mcq": "Who published the book 'The Organization of Behavior' in 1949, introducing a theoretical neural 

In [44]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:1996
Prompt Tokens:1510
Completion Tokens:486
Total Cost:0.001484


In [45]:
response

{'text': 'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]\n\nThe earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb\'s model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied human cognitive s

In [47]:
quiz=response.get("quiz")

In [49]:
quiz=json.loads(quiz)

In [50]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option} : {option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})
    

In [51]:
quiz_table_data

[{'MCQ': 'What term was coined in 1959 by Arthur Samuel to describe the field of computer gaming and artificial intelligence?',
  'Choices': 'a : Artificial Intelligence | b : Machine Learning | c : Neural Networks | d : Deep Learning',
  'Correct': 'b'},
 {'MCQ': "Who published the book 'The Organization of Behavior' in 1949, introducing a theoretical neural structure?",
  'Choices': 'a : Arthur Samuel | b : Donald Hebb | c : Walter Pitts | d : Warren McCulloch',
  'Correct': 'b'},
 {'MCQ': "Which company developed an experimental 'learning machine' called Cybertron in the early 1960s?",
  'Choices': 'a : IBM | b : Microsoft | c : Raytheon Company | d : Google',
  'Correct': 'c'},
 {'MCQ': 'According to Tom M. Mitchell, what is the definition of a computer program learning from experience?',
  'Choices': 'a : Improving performance at tasks with respect to some class of tasks | b : Repeating the same tasks over and over | c : Not learning from experience | d : Ignoring performance meas

In [52]:
quiz=pd.DataFrame(quiz_table_data)

In [53]:
quiz

,MCQ,Choices,Correct
0,What term was coined in 1959 by Arthur Samuel ...,a : Artificial Intelligence | b : Machine Lear...,b
1,Who published the book 'The Organization of Be...,a : Arthur Samuel | b : Donald Hebb | c : Walt...,b
2,Which company developed an experimental 'learn...,a : IBM | b : Microsoft | c : Raytheon Company...,c
3,"According to Tom M. Mitchell, what is the defi...",a : Improving performance at tasks with respec...,a
4,Which type of Machine Learning algorithm focus...,a : Supervised Learning Algorithms | b : Unsup...,c


In [54]:
quiz.to_csv("machineLearning.csv", index=False)